In [1]:
import pandas as pd
import numpy as np

In [27]:
df = pd.read_csv("C:/Users/akika/OneDrive/デスクトップ/secom-yield-optimization/secom_data.csv.csv")

In [28]:
df.shape

(1567, 592)

In [26]:
df.head()

,Time,0,1,2,3,4,5,6,7,8,...,581,582,583,584,585,586,587,588,589,Pass/Fail
0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1


## Finding Missing Values

In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1567 entries, 0 to 1566
Columns: 592 entries, Time to Pass/Fail
dtypes: float64(590), int64(1), str(1)
memory usage: 7.1 MB


In [31]:
df['Pass/Fail'].value_counts()

Pass/Fail
-1    1463
 1     104
Name: count, dtype: int64

In [32]:
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

292    1429
157    1429
158    1429
293    1429
492    1341
       ... 
585       1
586       1
587       1
588       1
589       1
Length: 538, dtype: int64

In [35]:
missing_pct = (df.isnull().sum()/(len(df))) * 100
missing_pct[missing_pct > 0].sort_values(ascending=False)

292    91.193363
157    91.193363
158    91.193363
293    91.193363
492    85.577537
         ...    
585     0.063816
586     0.063816
587     0.063816
588     0.063816
589     0.063816
Length: 538, dtype: float64

## Step1 Handling Missing Values
- Dropped sensors with >60% missing rate
- Investigating remaining missing value distribution

In [41]:
threshold = 60 # drop whose missing rate > 60%
cols_to_drop = missing_pct[missing_pct > threshold].index
df_cleaned = df.drop(columns=cols_to_drop)
df_cleaned.shape

(1567, 568)

In [43]:
remaining_missing = df_cleaned.isnull().sum()
remaining_missing[remaining_missing > 0].sort_values(ascending=False)

345    794
346    794
72     794
73     794
385    715
      ... 
32       1
558      1
559      1
588      1
589      1
Length: 514, dtype: int64

In [44]:
remaining_missing.describe()

count    568.000000
mean      26.160211
std       99.789659
min        0.000000
25%        1.000000
50%        6.000000
75%        9.000000
max      794.000000
dtype: float64

In [45]:
remaining_missing[remaining_missing > 9].sort_values(ascending=False)

73     794
72     794
346    794
345    794
385    715
      ... 
219     12
155     10
19      10
428     10
290     10
Length: 107, dtype: int64

## Step2 Missing Value Distribution Analysis

- 568 sensors remain after dropping >60% missing columns
- Distribution is right-skewed: mean (26) >> median (6)
- 107 sensors still have >9 missing values (75th percentile)
- Remaining ~461 sensors have minimal missing 

In [47]:
# Categorize columns by missing severity, based on missing count
bins = [0, 50, 200, 500, 1567]
labels = ['low (<=50)', 'medium (51-200)', 'high (200-500)', 'very high (>=500)']

# Sorts each value into the bin range it falls into
severity = pd.cut(remaining_missing[remaining_missing > 9], bins = bins, labels =labels)

# result
severity.value_counts()

low (<=50)           71
high (200-500)       20
medium (51-200)       8
very high (>=500)     8
Name: count, dtype: int64

## Missing Severity Breakdown (for 107 sensors with >9 missing)

| Category | Missing Range | Count | Strategy |
|---|---|---|---|
| Low | ≤50 | 71 | Median imputation only |
| Medium | 51-200 | 8 | Median imputation + missing flag |
| High | 201-500 | 20 | Median imputation + missing flag |
| Very High | >500 | 8 | Median imputation + missing flag (borderline drop candidates) |

In [82]:
# Sensors with more than 50 missing values get a "was_missing" flag,
# since missingness itself might carry signal (e.g. skipped process step)
flag_threshold = 50
cols_needing_flag = remaining_missing[remaining_missing > flag_threshold].index

for col in cols_needing_flag:
    df_cleaned[f'{col}_was_missing'] = df_cleaned[col].isnull().astype(int)

flag_cols = [c for c in df_cleaned.columns if c.endswith('_was_missing')]

print(f"Created {len(cols_needing_flag)} missing-indicator flag columns")
print(flag_cols)

Created 36 missing-indicator flag columns
['72_was_missing', '73_was_missing', '89_was_missing', '90_was_missing', '112_was_missing', '224_was_missing', '225_was_missing', '247_was_missing', '345_was_missing', '346_was_missing', '362_was_missing', '363_was_missing', '385_was_missing', '496_was_missing', '497_was_missing', '519_was_missing', '546_was_missing', '547_was_missing', '548_was_missing', '549_was_missing', '550_was_missing', '551_was_missing', '552_was_missing', '553_was_missing', '554_was_missing', '555_was_missing', '556_was_missing', '557_was_missing', '562_was_missing', '563_was_missing', '564_was_missing', '565_was_missing', '566_was_missing', '567_was_missing', '568_was_missing', '569_was_missing']


## Step 3: Impute Remaining Missing Values

Apply median imputation to all sensor columns. Median is used instead 
of mean because sensor data is prone to outliers, and median is more 
robust to extreme values.

In [77]:
from sklearn.impute import SimpleImputer

# Identify sensor columns only — exclude label, timestamp, 
# and the flag columns we just created (no need to impute)
sensor_cols = [c for c in df_cleaned.columns
               if not c.endswith('_was_missing')
               and c not in ['Pass/Fail', 'Time']]

# median is more rubust
imputer = SimpleImputer(strategy = 'median')
df_cleaned[sensor_cols] = imputer.fit_transform(df_cleaned[sensor_cols])

# confirm no missing value remaining
print(f"Shape:{df_cleaned.shape}")
print(f"Total missing value:{df_cleaned.isnull().sum().sum()}")

Shape:(1567, 604)
Total missing value:0


In [109]:
# Save the cleaned dataframe so 02_modeling.ipynb can load it 
df_cleaned.to_csv("C:/Users/akika/OneDrive/デスクトップ/secom-yield-optimization/notebooks/data_cleaned.csv")
print("Saved cleaned data")

Saved cleaned data
